# Notebook 04 — Governance Extractor and Incident Node Demo

**Author:** Arbaz Khalid — Safety & Governance Module  
**Status:** Rule-based baseline for entity extraction, wired to the Safety classifier

## Purpose
Take a report, decide if it is a hazard (Safety), extract what and where (Governance), look up a protocol, and output an incident node.

## Known limitations
- The extractor is rule-based (regex). The final system would use a fine-tuned model for this.
- The protocol lookup is a **placeholder dictionary**, not a real RAG pipeline over NYC documents.
- Text-only reports have no coordinates yet; geocoding is a later step.

In [ ]:
import re
import json
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

df = pd.read_csv("data/processed/nyc311_ida_labeled.csv").dropna(subset=["text", "label"])
vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2)
clf = LogisticRegression(max_iter=1000, class_weight="balanced").fit(vec.fit_transform(df["text"]), df["label"])
print("Safety model ready:", list(clf.classes_))

In [ ]:
STREET_WORDS = r"(?:street|st|avenue|ave|boulevard|blvd|road|rd|drive|dr|parkway|pkwy|place|pl)"
ADDRESS_RE = re.compile(rf"\b\d{{1,5}}(?:-\d{{1,5}})?\s+(?:[A-Za-z0-9]+\s+){{0,3}}{STREET_WORDS}\b", re.I)
STREET_RE = re.compile(rf"\b(?:[A-Za-z]+|\d{{1,3}}(?:st|nd|rd|th)?)\s+{STREET_WORDS}\b", re.I)
SKIP_FIRST = {"the", "a", "an", "my", "our", "this", "that", "every", "same", "main", "next"}
BOROUGHS = ["manhattan", "brooklyn", "queens", "bronx", "staten island"]
SEVERE = ["waist", "submerged", "stuck", "trapped", "emergency", "rescue", "cars", "river"]

def extract_entities(text):
    location = None
    m = ADDRESS_RE.search(text)
    if m:
        location = m.group(0)
    else:
        for m in STREET_RE.finditer(text):
            if m.group(0).split()[0].lower() not in SKIP_FIRST:
                location = m.group(0)
                break
    borough = next((b.title() for b in BOROUGHS if b in text.lower()), None)
    severity = "high" if any(w in text.lower() for w in SEVERE) else "medium"
    return {"location_text": location, "borough": borough, "severity": severity}

In [ ]:
SOP = {   # PLACEHOLDER text for the demo. Real version = RAG over NYC protocol documents.
    "flood": "SOP-FLOOD-01 (placeholder): dispatch DEP crew, close affected road segment",
    "road_blockage": "SOP-BLOCK-01 (placeholder): notify DOT, reroute emergency vehicles",
}

def build_incident(text, source, timestamp, lat=None, lon=None):
    probs = clf.predict_proba(vec.transform([text]))[0]
    label = clf.classes_[probs.argmax()]
    if label == "non_hazard":
        return None                      # Safety gate: discard
    ent = extract_entities(text)
    return {
        "incident_id": f"INC-{abs(hash(text)) % 10**6:06d}",
        "source": source,
        "timestamp": str(timestamp),
        "hazard_type": label,
        "severity": ent["severity"],
        "confidence": round(float(probs.max()), 2),
        "location_text": ent["location_text"],
        "borough": ent["borough"],
        "lat": lat,
        "lon": lon,
        "protocol": SOP[label],
        "verified_by_rag": False,        # becomes True once real RAG is built
    }

In [ ]:
print("--- From 311 reports (have coordinates) ---")
for _, r in df[df["label"] != "non_hazard"].sample(3, random_state=3).iterrows():
    print(json.dumps(build_incident(r["text"], "NYC_311", r["created_date"], r["latitude"], r["longitude"]), indent=2))

print("\n--- From hand-written citizen text (no coordinates) ---")
for t in ["water is rushing down 5th avenue in Manhattan, cars are stuck",
          "huge tree fell across Queens Blvd, nobody can pass",
          "my neighbor is playing loud music again"]:
    node = build_incident(t, "SOCIAL_DEMO", "2021-09-01T22:00:00")
    print(t, "->", json.dumps(node, indent=2) if node else "DISCARDED (non_hazard)")